# 기본 LLM의 연산별 Roofline: PyTorch (GPU) · JAX (TPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jake-Song/tensor2silicon/blob/main/notebooks/llm_roofline.ipynb)

LLaMA 형태의 작은 decoder-only LLM을 **PyTorch와 JAX로 똑같이** 만들고, 한 층을 이루는 연산을 하나씩 따로 재서 각 연산이 **compute-bound인지 memory-bound인지** roofline으로 판정한다.

[① 모델 → 연산](../docs/01-model-to-ops.md)에서 본 것처럼 Transformer 한 층은 **9개의 matmul**과 그 사이의 작은 연산으로 이뤄진다.

| # | matmul | 모양 (prefill) | 가중치 |
|---|---|---|---|
| 1–3 | `q_proj`, `k_proj`, `v_proj` | `[BT, D] @ [D, NH]` | 있음 |
| 4 | `qk` = Q·Kᵀ | `B·N × [T, H] @ [H, S]` | 없음 |
| 5 | `pv` = P·V | `B·N × [T, S] @ [S, H]` | 없음 |
| 6 | `o_proj` | `[BT, NH] @ [NH, D]` | 있음 |
| 7–9 | `gate_proj`, `up_proj`, `down_proj` | `[BT, D] @ [D, F]`, `[BT, F] @ [F, D]` | 있음 |

사이에 있는 연산: `LayerNorm`(×2), `RoPE + head 분리`, `mask + softmax`, `head 병합`, `SwiGLU`, residual add(×2), 그리고 decode에서는 KV cache 쓰기.

판정 기준은 [Compute Bound와 Memory Bound](../docs/00-compute-memory-bound.md)의 arithmetic intensity다.

$$\text{AI} = \frac{\text{FLOPs}}{\text{bytes}}, \qquad \text{AI} < \text{ridge} = \frac{\text{peak FLOP/s}}{\text{peak bytes/s}} \Rightarrow \text{memory-bound}$$

두 단계를 비교한다.

- **prefill**: 토큰 2048개를 한 번에 처리 (B=1, T=2048). 가중치 matmul의 AI ≈ 수백 → compute-bound 예상
- **decode**: 새 토큰 1개, KV cache 2048개 (B=8, T=1, S=2048). 같은 matmul의 AI ≈ B = 8 → memory-bound 예상

**런타임**: GPU(A100 권장)면 PyTorch, TPU(v5e / v6e)면 JAX 버전이 실행된다. 코드는 [`llm_roofline/`](../llm_roofline)에 있다.

## 0. 런타임 확인

In [ ]:
import importlib.util, subprocess

RUNTIME = "cpu"
if importlib.util.find_spec("torch"):
    import torch
    if torch.cuda.is_available():
        RUNTIME = "gpu"
        p = torch.cuda.get_device_properties(0)
        print("torch", torch.__version__, "|", p.name, "| SMs", p.multi_processor_count, "| HBM", round(p.total_memory / 2**30, 1), "GiB")
if RUNTIME == "cpu":
    # JAX는 import하지 않고 하위 프로세스로 확인한다. 이 커널이 TPU를 잡으면 벤치마크 프로세스가 TPU를 못 쓴다.
    r = subprocess.run(["python", "-c", "import jax; d = jax.devices()[0]; print(d.platform, d.device_kind)"],
                       capture_output=True, text=True)
    platform, _, kind = r.stdout.strip().partition(" ")
    if platform == "tpu":
        RUNTIME = "tpu"
        print("TPU:", kind)
print("RUNTIME =", RUNTIME, "->", {"gpu": "PyTorch (torch_llm)", "tpu": "JAX (jax_llm)", "cpu": "JAX on CPU, tiny preset only"}[RUNTIME])

## 1. 저장소 받기

In [ ]:
import os, sys
REPO = "/content/tensor2silicon"
if not os.path.exists(REPO):
    !git clone https://github.com/Jake-Song/tensor2silicon.git {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}
sys.path.insert(0, REPO)
!ls llm_roofline

## 2. 벤치마크 실행

각 연산을 **따로** 실행해서 잰다.

- **캐시 우회**: 연산마다 입력 사본을 여러 벌(총 ≥ 512 MB) 만들어 돌려 가며 쓴다. 그래야 가중치가 A100의 40 MB L2에 남아 있지 않고 매번 HBM에서 읽힌다.
- **PyTorch**: 연산을 CUDA graph로 캡처해 kernel 시간만 잰다. CPU launch 오버헤드는 뺀다.
- **JAX**: 연산을 `lax.scan`으로 k번 반복하는 jit 함수 하나로 만들어 dispatch 오버헤드를 뺀다. `cost_analysis()`로 XLA가 센 FLOPs·bytes도 함께 보여준다(`xlaFLOP`, `xlaB` 열).
- **Roofline 천장**: spec sheet 값 대신 **직접 잰 값**을 쓴다(큰 matmul의 TFLOP/s, 1 GiB copy·read 중 빠른 쪽의 GB/s).

표 열: `AI` = FLOP/byte, `theory` = AI와 ridge 비교로 예측한 병목, `%roof` = roofline 시간 / 측정 시간, `measured` = 측정에서 더 많이 쓴 자원. roofline 시간이 5 µs 미만이고 %roof가 25% 미만이면 `latency`(launch·dispatch 지연이 지배)로 표시한다.

A100 기준으로 몇 분 걸린다(`torch.compile` 포함). TPU는 연산마다 XLA 컴파일이 있어 조금 더 걸린다.

In [ ]:
PRESET = "tiny" if RUNTIME == "cpu" else "colab"   # "small"은 8 GB급 GPU용
OUT = f"results_{RUNTIME}.json"

if RUNTIME == "gpu":
    !python -m llm_roofline.torch_llm --preset {PRESET} --out {OUT} --no-profile
else:
    !python -m llm_roofline.jax_llm --preset {PRESET} --out {OUT}

## 3. Roofline 그림

x축은 arithmetic intensity, y축은 달성한 TFLOP/s다. 굵은 선이 roofline `min(peak FLOP/s, AI × BW)`이고, 세로선이 ridge point다.

- 선 **가까이** 있는 점은 하드웨어 한계까지 쓰고 있다.
- ridge **왼쪽**(대각선 아래)은 memory-bound, **오른쪽**(수평선 아래)은 compute-bound 영역이다.
- 같은 모양의 matmul(`q,k,v,o_proj`처럼)은 점 하나로 합쳤다. 데이터 이동만 하는 연산(`embed`, `merge_heads`, KV cache 쓰기)은 FLOPs가 0이라 log 축에 그릴 수 없어서 아래 표에만 나온다.

In [ ]:
import matplotlib.pyplot as plt
from llm_roofline.plot import plot_roofline, plot_time_breakdown, summary_table

plot_roofline(OUT)
plt.show()

**읽는 법**: prefill 패널에서 `q,k,v,o_proj`·`gate,up_proj`·`down_proj`·`lm_head`는 AI가 수백~천이라 ridge 오른쪽 수평선 근처에 있다(compute-bound). decode 패널에서는 **같은 matmul**이 AI ≈ 8로 떨어져 대각선 위에 붙는다(memory-bound). 토큰 하나를 만들 때마다 가중치 전체를 HBM에서 다시 읽기 때문이다. `LayerNorm`·`softmax`·elementwise는 두 단계 모두 AI ≈ 1 이하라 항상 memory-bound다.

## 4. 연산별 표

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)
pd.DataFrame(summary_table(OUT)).set_index(["phase", "op"])

## 5. 시간은 어디에 쓰이나

연산별 시간 × 호출 횟수(층 수 L=8)를 합쳐 forward 한 번에 각 연산이 차지하는 시간을 본다.

In [ ]:
plot_time_breakdown(OUT)
plt.show()

## 6. 모델 전체: 연산을 합치면(fusion) 얼마나 빨라지나

- **sum of ops**: 위에서 따로 잰 연산 시간 × 호출 횟수의 합. 연산 사이에 fusion이 전혀 없을 때의 시간이다.
- **PyTorch**: `eager`(Python에서 연산 하나씩 launch), `eager + CUDA graph`(같은 kernel, launch 오버헤드만 제거), `torch.compile`(Inductor가 elementwise·reduction을 Triton kernel로 합침)
- **JAX**: `jax.jit`으로 모델 전체를 XLA 프로그램 하나로 컴파일

In [ ]:
import json
res = json.load(open(OUT))
for ph, v in res["phases"].items():
    m, s = v["model"], v["summary"]
    parts = [f"{k.removesuffix('_s')} {x * 1e3:.2f} ms" for k, x in m.items() if k.endswith("_s") and k != "compile_time_s"]
    print(f"{ph:8s} sum of ops {s['sum_of_ops_s'] * 1e3:.2f} ms | " + " | ".join(parts))

## 7. Attention: 따로 계산 vs fused kernel

모델 안의 attention은 일부러 `qk → mask+softmax → pv` 세 연산으로 나눠 두었다. 그러면 `[B, N, T, S]` 크기의 score 행렬이 HBM에 두 번 쓰이고 두 번 읽힌다. fused kernel(PyTorch `F.scaled_dot_product_attention`, JAX `jax.nn.dot_product_attention` 또는 TPU Pallas flash attention)은 score를 on-chip 메모리에만 두기 때문에 이 트래픽이 없다. 자세한 내용은 [FlashAttention](../docs/flash-attention.md) 문서 참고.

In [ ]:
for ph, v in res["phases"].items():
    a = v["attention"]
    print(f"{ph:8s} unfused {a['unfused_s'] * 1e6:8.1f} us | {a['fused_impl']} {a['fused_s'] * 1e6:8.1f} us "
          f"| {a['unfused_s'] / a['fused_s']:.2f}x | max |diff| {a['max_abs_err']:.2g}")

## 8. 참고 결과: A100 (PyTorch) · TPU v5e (JAX)

저장소의 `llm_roofline/results/`에 Colab에서 미리 잰 결과가 있다. 같은 모델·같은 크기(`colab` preset, bf16)다. 런타임을 바꾸지 않고도 두 하드웨어를 비교할 수 있다.

In [ ]:
REF = ["llm_roofline/results/colab_a100.json", "llm_roofline/results/colab_tpu_v5e.json"]
for path in REF:
    plot_roofline(path)
plt.show()

### 관찰 (위 참고 결과 기준)

| | A100 · PyTorch | TPU v5e · JAX |
|---|---|---|
| 측정 천장 | 261 TFLOP/s · 1433 GB/s (ridge ≈ 182) | 185 TFLOP/s · 746 GB/s (ridge ≈ 248) |
| prefill `q,k,v,o_proj` (2048³) | 162 TFLOP/s (**62%**) | 163 TFLOP/s (**88%**) |
| prefill `gate,up,down_proj` | 173–219 TFLOP/s (66–84%) | 165–171 TFLOP/s (89–92%) |
| decode `q,k,v,o_proj` (AI ≈ 8) | 709 GB/s (49%) | 619 GB/s (83%) |
| decode `gate,up,down`, `lm_head` | 986–1284 GB/s (69–90%) | 699–735 GB/s (94–99%) |
| prefill `mask+softmax` | 전체의 **30%** | 전체의 **24%** |
| 모델 전체 prefill | eager 22.6 ms → `torch.compile` 16.6 ms | 연산 합 25.7 ms → `jax.jit` 17.9 ms |
| 모델 전체 decode | eager 5.50 ms → CUDA graph 2.51 ms | 연산 합 3.28 ms → `jax.jit` 2.96 ms |

1. **같은 matmul, 다른 병목**: prefill에서 compute-bound였던 7개의 가중치 matmul이 decode에서는 AI ≈ 8로 떨어져 모두 memory-bound가 된다. decode를 빠르게 하는 방법은 FLOPs 최적화가 아니라 **바이트 줄이기**(batch 키우기, weight quantization)다.
2. **`qk`·`pv`는 prefill에서도 memory-bound**다(AI ≈ 114 < ridge). score 행렬 `[16, 2048, 2048]`을 HBM에 쓰고 다시 읽기 때문이다. 이것이 FlashAttention이 score를 HBM에 쓰지 않는 이유다. A100에서는 fused SDPA가 unfused보다 **7.5배** 빨랐다.
3. **softmax는 FLOPs가 거의 없는데도 prefill 시간의 1/4~1/3**을 쓴다. 전형적인 memory-bound 연산이고 fusion의 첫 번째 대상이다.
4. **A100의 2048×2048 matmul이 62%에 그치는 이유**는 아마 tile 배치(wave quantization)다. 출력 타일 수가 SM 108개로 딱 나눠지지 않으면 마지막 wave에서 SM 대부분이 논다. 같은 모양이 TPU에서 88%인 것과 대비된다. `torch.profiler`로 kernel 이름과 grid 크기를 보며 확인해 볼 만한 가설이다.
5. **decode의 작은 연산은 `latency`로 판정**된다(LayerNorm·RoPE·softmax 등). 데이터가 수십 KB라 roofline 시간이 1 µs도 안 되는데, 실제로는 kernel 하나 띄우는 데 수 µs가 든다. A100에서 eager decode 5.50 ms가 CUDA graph로 2.51 ms가 되는 것이 이 launch 오버헤드다.
6. **100%를 넘는 점**: A100의 `embed`(118%)는 gather하는 행 32 MB가 L2(40 MB)에 남아서, `residual`(103%)은 측정 천장 자체의 오차 범위다. roofline 천장은 측정값이라 spec sheet(1555 GB/s)보다 낮다.
7. **TPU v5e에서 Pallas flash attention이 실패**한 것(`Unsupported version: expected <= 7 but got 8`)은 Colab의 jax와 libtpu 버전이 맞지 않아서다. 이때는 `jax.nn.dot_product_attention`(XLA 구현)으로 대신 비교하므로 prefill에서 fused와 unfused가 비슷하게 나온다.